# HomeRehab AI — Fit3D landmark extraction (Gate 2.0)

Runs MediaPipe Pose **Lite** (the model the app ships) on Fit3D videos of the exercises our packs cover, and saves one small `.npz` per subject × exercise × camera: MediaPipe landmarks plus the Vicon joints in 2D (projected into that camera) and 3D. The analysis then runs on the laptop: `ml/eval/fit3d_gate_eval.py`.

**Licence:** Fit3D is for non-commercial research only and is licensed to *you personally*. Do not share the download link, the data or these `.npz` files outside the project.

**Steps:** Runtime → Run all. It asks for Google Drive access (to save the results) and for your Fit3D download link in cell 3. CPU runtime is enough.

In [ ]:
# 1. Tools and the pose model (same Lite model as the app)
!pip -q install mediapipe opencv-python-headless
!wget -q -nc -O /content/pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
import mediapipe as mp, cv2, numpy as np, json, glob, os, time
print('mediapipe', mp.__version__)

In [ ]:
# 2. Results go to Google Drive so a disconnect does not lose them (already-done videos are skipped)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/homerehab-fit3d/landmarks'
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT)

In [ ]:
# 3. Download the Fit3D *Training Set* (18 GB) into this Colab machine (not into Drive)
# On fit3d.imar.ro → Download, right-click "Training Set" → Copy link address, and paste it below.
FIT3D_TRAIN_URL = ''  #@param {type:"string"}
ZIP = '/content/fit3d_train.zip'
assert FIT3D_TRAIN_URL, 'paste the Training Set link into FIT3D_TRAIN_URL first'
!wget -c -q --show-progress -O {ZIP} "{FIT3D_TRAIN_URL}"
with open(ZIP, 'rb') as f:
    magic = f.read(4)
size_gb = os.path.getsize(ZIP) / 1e9
print(f'downloaded {size_gb:.1f} GB')
assert magic.startswith(b'PK'), ('Not a zip file: the link needs your browser login. '
                                 'Tell Claude; the fallback is downloading on the laptop after freeing space.')

In [ ]:
# 4. Unzip only what we need: videos, 3D joints, camera parameters
!unzip -q -n {ZIP} '*/videos/*' '*/joints3d_25/*' '*/camera_parameters/*' -d /content/fit3d
TRAIN = sorted(glob.glob('/content/fit3d/**/train', recursive=True))[0]
SUBJECTS = sorted(os.listdir(TRAIN))
ACTIONS = sorted({os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{TRAIN}/*/joints3d_25/*.json')})
print('subjects:', SUBJECTS)
print(len(ACTIONS), 'actions:', ACTIONS)

In [ ]:
# 5. Keep the exercises our packs measure (same keywords as ml/eval/fit3d_gate_eval.py)
KEYWORDS = ('lunge', 'squat', 'lateral_raise', 'side_raise', 'front_raise')
CHOSEN = [a for a in ACTIONS if any(k in a.lower().replace(' ', '_').replace('-', '_') for k in KEYWORDS)]
print('chosen:', CHOSEN)
STEP = 3  # every 3rd frame (Fit3D is 50 fps, so about 17 fps: enough for rep shapes, 3x faster)

In [ ]:
# 6. Extract. About 1-2 minutes per video on a CPU runtime; progress is printed.
# Projection follows the official Fit3D tools (imar_vision_datasets_tools/util/dataset_util.py).
def project(points3d, intr):
    p = np.asarray(intr['p'])[:, [1, 0]]
    x = points3d[:, :2] / points3d[:, 2:3]
    r2 = np.sum(x ** 2, axis=1)
    radial = 1 + np.transpose(np.matmul(np.asarray(intr['k']), np.array([r2, r2 ** 2, r2 ** 3])))
    tan = np.matmul(x, np.transpose(p))
    xx = x * (tan + radial) + r2[:, None] * p
    return np.asarray(intr['f']) * xx + np.asarray(intr['c'])

def landmarker():
    o = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path='/content/pose_landmarker_lite.task'),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_poses=1)
    return mp.tasks.vision.PoseLandmarker.create_from_options(o)

jobs = [(s, a, c) for s in SUBJECTS for a in CHOSEN
        for c in sorted(os.listdir(f'{TRAIN}/{s}/videos')) if os.path.exists(f'{TRAIN}/{s}/videos/{c}/{a}.mp4')]
print(len(jobs), 'videos to process')
start = time.time()
for n, (subj, action, cam) in enumerate(jobs, 1):
    out = f"{OUT}/{subj}_{action.replace(' ', '_')}_{cam}.npz"
    if os.path.exists(out):
        continue
    j3d = np.asarray(json.load(open(f'{TRAIN}/{subj}/joints3d_25/{action}.json'))['joints3d_25'], dtype=float)
    cam_params = json.load(open(f'{TRAIN}/{subj}/camera_parameters/{cam}/{action}.json'))
    R, T = np.asarray(cam_params['extrinsics']['R']), np.asarray(cam_params['extrinsics']['T'])
    cap = cv2.VideoCapture(f'{TRAIN}/{subj}/videos/{cam}/{action}.mp4')
    fps = cap.get(cv2.CAP_PROP_FPS) or 50.0
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames, marks, gt2d, gt3d = [], [], [], []
    with landmarker() as lm:
        i = 0
        while i < len(j3d):
            ok, bgr = cap.read()
            if not ok:
                break
            if i % STEP == 0:
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                res = lm.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(i * 1000 / fps))
                m = np.full((33, 5), np.nan)
                if res.pose_landmarks:
                    for k, p in enumerate(res.pose_landmarks[0]):
                        m[k] = (p.x * width, p.y * height, (p.z or 0.0) * width, p.visibility or 0.0, p.presence or 0.0)
                cam_pts = np.matmul(j3d[i] - T, np.transpose(R))
                frames.append(i); marks.append(m); gt3d.append(j3d[i])
                gt2d.append(project(cam_pts, cam_params['intrinsics_w_distortion']))
            i += 1
    cap.release()
    meta = dict(subject=subj, action=action, camera=cam, fps=fps / STEP, width=width, height=height)
    np.savez_compressed(out, frames=np.asarray(frames), marks=np.asarray(marks, dtype=np.float32),
                        gt2d=np.asarray(gt2d, dtype=np.float32), gt3d=np.asarray(gt3d, dtype=np.float32),
                        meta=json.dumps(meta))
    left = (time.time() - start) / n * (len(jobs) - n)
    print(f'{n}/{len(jobs)} {subj} {action} {cam}: {len(frames)} frames, about {left / 60:.0f} min left')
print('done')

In [ ]:
# 7. One zip to download to the laptop → unzip into Grad_1/data/derived/fit3d-landmarks/
!cd {OUT} && zip -q -r /content/drive/MyDrive/homerehab-fit3d/fit3d-landmarks.zip .
print('Download from Drive: MyDrive/homerehab-fit3d/fit3d-landmarks.zip')